In [1]:
from pathlib import Path
import re
import json
import time
from collections import defaultdict
import pandas as pd
import requests
import certifi
import pdfplumber
from bs4 import BeautifulSoup

# assumes notebook is inside govresponseai/notebooks/
BASE = Path.cwd().resolve().parent
METADATA = BASE / "metadata"
RAW = BASE / "data_raw"
PROCESSED = BASE / "data_processed"
RESULTS = BASE / "results"

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# 1 paths
doc_index_path = METADATA / "doc_index.csv"
df = pd.read_csv(doc_index_path)

print("BASE:", BASE)
print("doc_index exists:", doc_index_path.exists())
print("df shape:", df.shape)
display(df)

BASE: /Users/wukezhang416/govresponseai
doc_index exists: True
df shape: (18, 8)


,doc_id,country,title,issuing_body,date,url,format,notes
0,us_doc1,US,AI Risk Management Framework 1.0,NIST,2023-01-26,https://nvlpubs.nist.gov/nistpubs/ai/nist.ai.1...,pdf,core framework
1,us_doc2,US,Blueprint for an AI Bill of Rights,OSTP/White House,2022-10-04,https://marketingstorageragrs.blob.core.window...,pdf,principles doc
2,us_doc3,US,OMB M-24-10: AI Governance & Risk Mgmt,OMB,2024-03-28,https://www.whitehouse.gov/wp-content/uploads/...,pdf,agency AI use governance memo
3,us_doc4,US,OMB M-24-18: Responsible AI Acquisition,OMB,2024-09-24,https://www.whitehouse.gov/wp-content/uploads/...,pdf,AI procurement memo
4,us_doc5,US,NIST AI 600-1: GenAI Profile,NIST,2024-07-01,https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.6...,pdf,GenAI risk profile companion to AI RMF
5,us_doc6,US,NIST AI RMF Playbook,NIST,2023-03-30,https://airc.nist.gov/docs/AI_RMF_Playbook.pdf,pdf,implementation playbook
6,us_doc7,US,NIST SP 800-218 SSDF,NIST,2022-02-03,https://nvlpubs.nist.gov/nistpubs/specialpubli...,pdf,governance/assurance framework baseline
7,cn_algo_2022,CN,互联网信息服务算法推荐管理规定,CAC+MIIT+MPS+SAMR,2021-12-31,https://www.cac.gov.cn/2022-01/04/c_1642894606...,html,algorithm recommendation regulation
8,cn_genai_2023,CN,生成式人工智能服务管理暂行办法,CAC,2023-07-13,https://www.cac.gov.cn/2023-07/13/c_1690898327...,html,interim measures for genAI services
9,cn_deepsynthesis_2022,CN,互联网信息服务深度合成管理规定,CAC+MIIT+MPS,2022-12-11,https://www.cac.gov.cn/2022-12/11/c_1672221949...,html,deep synthesis regulation


In [2]:
# 2 helpers
def safe_filename(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9\u4e00-\u9fff._-]+", "_", str(s)).strip("_")
    return s[:150] if len(s) > 150 else s

def download_file(url: str, out_path: Path, timeout=60):
    if out_path.exists() and out_path.stat().st_size > 0:
        return "cached"

    r = requests.get(
        url,
        timeout=timeout,
        headers={"User-Agent": "Mozilla/5.0"},
        verify=certifi.where()
    )
    r.raise_for_status()
    out_path.write_bytes(r.content)
    return "downloaded"

def parse_pdf_to_text(pdf_path: Path) -> str:
    texts = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            t = page.extract_text() or ""
            t = t.replace("\x00", " ").strip()
            if t:
                texts.append(t)
    return "\n".join(texts)

def parse_html_to_text(html_path: Path) -> str:
    html = html_path.read_text(encoding="utf-8", errors="ignore")

    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text("\n")
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return "\n".join(lines)

def chunk_text(text: str, max_chars=500):
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    chunks = []
    cur = ""

    for p in paras:
        if len(cur) + len(p) + 1 <= max_chars:
            cur = (cur + "\n" + p).strip()
        else:
            if cur:
                chunks.append(cur)
            cur = p

    if cur:
        chunks.append(cur)

    return chunks


In [3]:
# 3 main loop
records_by_country = defaultdict(list)
failed_docs = []

required_cols = ["doc_id", "country", "title", "issuing_body", "date", "url", "format"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"doc_index.csv is missing columns: {missing_cols}")

for _, row in df.iterrows():
    doc_id = str(row["doc_id"]).strip()
    country = str(row["country"]).strip().upper()
    title = str(row["title"]).strip()
    issuing = str(row["issuing_body"]).strip()
    date = str(row["date"]).strip()
    url = str(row["url"]).strip()
    fmt = str(row["format"]).strip().lower()

    if fmt not in ["pdf", "html"]:
        print(f"[SKIP] {doc_id}: unsupported format = {fmt}")
        failed_docs.append({"doc_id": doc_id, "reason": f"unsupported format {fmt}"})
        continue

    suffix = "pdf" if fmt == "pdf" else "html"
    fname = safe_filename(f"{doc_id}_{title}") or doc_id
    raw_path = RAW / f"{fname}.{suffix}"

    try:
        status = download_file(url, raw_path)
        size_kb = raw_path.stat().st_size / 1024
        print(f"[{country}] {doc_id}: {status} -> {raw_path.name} ({size_kb:.1f} KB)")

        if fmt == "pdf":
            text = parse_pdf_to_text(raw_path)
        else:
            text = parse_html_to_text(raw_path)

        if not text.strip():
            raise ValueError("parsed text is empty")

        chunks = chunk_text(text, max_chars=500)

        for i, ch in enumerate(chunks):
            records_by_country[country].append({
                "doc_id": doc_id,
                "chunk_id": f"{doc_id}_{i:04d}",
                "country": country,
                "title": title,
                "issuing_body": issuing,
                "date": date,
                "url": url,
                "format": fmt,
                "text": ch
            })

    except Exception as e:
        print(f"[ERROR] {doc_id}: {e}")
        failed_docs.append({"doc_id": doc_id, "reason": str(e)})

    time.sleep(1)


[US] us_doc1: cached -> us_doc1_AI_Risk_Management_Framework_1.0.pdf (1900.5 KB)
[US] us_doc2: cached -> us_doc2_Blueprint_for_an_AI_Bill_of_Rights.pdf (11401.3 KB)
[US] us_doc3: cached -> us_doc3_OMB_M-24-10_AI_Governance_Risk_Mgmt.pdf (518.1 KB)
[US] us_doc4: cached -> us_doc4_OMB_M-24-18_Responsible_AI_Acquisition.pdf (447.9 KB)
[US] us_doc5: cached -> us_doc5_NIST_AI_600-1_GenAI_Profile.pdf (1147.1 KB)
[US] us_doc6: cached -> us_doc6_NIST_AI_RMF_Playbook.pdf (2814.7 KB)
[US] us_doc7: cached -> us_doc7_NIST_SP_800-218_SSDF.pdf (722.5 KB)
[CN] cn_algo_2022: cached -> cn_algo_2022_互联网信息服务算法推荐管理规定.html (30.5 KB)
[CN] cn_genai_2023: cached -> cn_genai_2023_生成式人工智能服务管理暂行办法.html (28.7 KB)
[CN] cn_deepsynthesis_2022: cached -> cn_deepsynthesis_2022_互联网信息服务深度合成管理规定.html (29.0 KB)
[CN] cn_labeling_2025: cached -> cn_labeling_2025_人工智能生成合成内容标识办法.html (22.8 KB)
[CN] cn_cybersecuritylaw_2016: cached -> cn_cybersecuritylaw_2016_中华人民共和国网络安全法.html (19.3 KB)
[ERROR] cn_datasecuritylaw_2021: HTTPSCo

In [4]:

# 4 outputs
all_records = []

for country, recs in records_by_country.items():
    out_path = PROCESSED / f"{country.lower()}_chunks.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for rec in recs:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Saved {country}: {len(recs)} chunks -> {out_path}")
    all_records.extend(recs)

all_out_path = PROCESSED / "all_chunks.jsonl"
with all_out_path.open("w", encoding="utf-8") as f:
    for rec in all_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved ALL: {len(all_records)} chunks -> {all_out_path}")

failed_path = PROCESSED / "failed_docs.json"
with failed_path.open("w", encoding="utf-8") as f:
    json.dump(failed_docs, f, ensure_ascii=False, indent=2)

print(f"Failed docs logged to: {failed_path}")
print("Done.")

Saved US: 2481 chunks -> /Users/wukezhang416/govresponseai/data_processed/us_chunks.jsonl
Saved CN: 482 chunks -> /Users/wukezhang416/govresponseai/data_processed/cn_chunks.jsonl
Saved ALL: 2963 chunks -> /Users/wukezhang416/govresponseai/data_processed/all_chunks.jsonl
Failed docs logged to: /Users/wukezhang416/govresponseai/data_processed/failed_docs.json
Done.
